In [1]:
# Set up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(parent_dir)
task_name = 'MemoryCircuits_4s_5r_MAK_REINFORCE_SIL'
print('Working directory set to:', parent_dir)

Working directory set to: /local0/rossin/git/CRN-GenerativeAI


In [2]:
# Import general packages
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
from itertools import product
from tqdm import tqdm

# Import Agent-Environment packages
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_by_ordered_index import AddReactionByOrderedIndex
from RL4CRN.policies.add_reaction_by_index import AddReactionByIndex

# Import Interface packages
from RL4CRN.env2agent_interface.explicit_observer import ExplicitObserver
from RL4CRN.env2agent_interface.explicit_tensorizer import ExplicitTensorizer
from RL4CRN.agent2env_interface.library_actuator import LibraryActuator
from RL4CRN.agent2env_interface.iocrn_stepper import IOCRNStepper

# Import CRN packages
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.utils.ic import IC
from RL4CRN.iocrns.reaction_library import construct_mass_action_library

# Import Reward packages
from RL4CRN.rewards.deterministic import dynamic_tracking_error_piecewise
from RL4CRN.rewards.stochastic import dynamic_tracking_error_SSA

In [3]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "vhIR3uyqsKyU4L7SA8fLCfTSC"
logger = CometLogger(
    api_key=api_key,
    project=task_name,        
    workspace="redsnic", 
    name=f'{task_name}_{timestamp}',
)
logger = logger.experiment

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/redsnic/memorycircuits-4s-5r-mak-reinforce-sil/1ce583f9129841519c170c7a77fea1c2



In [4]:
# Construct the template CRN
# add dilution and production reactions for all species
productions = []
dilutions = []
n_inputs = 4

# the idea here is to have 3 species: X (input), Set (set/reset), Mem (memory)
# with respective inputs u_X, u_set, u_mem
# in a first time window we register the memory (using Set and Mem) which will determine if the output of our network will be the id(X) or not (NOT(X))

species_labels = ["X", "Set", "Mem", "OUT"] 

productions.append(MassAction(reactant_labels=[], product_labels=['X'], input_channels=['u_X'], params=[1.], params_controllability=[True]))
productions.append(MassAction(reactant_labels=[], product_labels=['Set'], input_channels=['u_set'], params=[1.], params_controllability=[True]))
productions.append(MassAction(reactant_labels=[], product_labels=['Mem'], input_channels=['u_mem'], params=[1.], params_controllability=[True]))

# we use no dilution for now

crn_template = IOCRN(productions + dilutions, output_labels=['OUT'])
crn_template.compile()
p = crn_template.num_inputs # Number of inputs of the IOCRNs
print("Template CRN:")
print(crn_template)

# Construct the library of possible reactions
library = construct_mass_action_library(species_labels=species_labels, order=2)
crn_template.set_library_context(library)
M = len(library.reactions) # Number of possible reactions
K = library.get_num_parameters() # Total number of parameters in all the reactions of the library
print("Library of possible reactions:")
print(library)
print("------------------------------------------------")

Template CRN:
Inputs: ['u_X', 'u_mem', 'u_set'] 
Species: ['Mem', 'OUT', 'Set', 'X'] 
Output Species: ['OUT'] 
∅ ----> X;  [MAK(1.0, u_X)]
∅ ----> Set;  [MAK(1.0, u_set)]
∅ ----> Mem;  [MAK(1.0, u_mem)]
Library of possible reactions:
Number of reactions: 211
R0: ∅ ----> ∅;  [MAK(None)]
R1: ∅ ----> X;  [MAK(None)]
R2: ∅ ----> Set;  [MAK(None)]
R3: ∅ ----> Mem;  [MAK(None)]
R4: ∅ ----> OUT;  [MAK(None)]
R5: ∅ ----> X + X;  [MAK(None)]
R6: ∅ ----> Set + X;  [MAK(None)]
R7: ∅ ----> Mem + X;  [MAK(None)]
R8: ∅ ----> OUT + X;  [MAK(None)]
R9: ∅ ----> Set + Set;  [MAK(None)]
R10: ∅ ----> Mem + Set;  [MAK(None)]
R11: ∅ ----> OUT + Set;  [MAK(None)]
R12: ∅ ----> Mem + Mem;  [MAK(None)]
R13: ∅ ----> Mem + OUT;  [MAK(None)]
R14: ∅ ----> OUT + OUT;  [MAK(None)]
R15: X ----> ∅;  [MAK(None)]
R16: X ----> Set;  [MAK(None)]
R17: X ----> Mem;  [MAK(None)]
R18: X ----> OUT;  [MAK(None)]
R19: X ----> X + X;  [MAK(None)]
R20: X ----> Set + X;  [MAK(None)]
R21: X ----> Mem + X;  [MAK(None)]
R22: X ----> OU

In [5]:
# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'Number of CPUs available: {os.cpu_count()}') 

Using device: cuda
Number of CPUs available: 128


In [6]:
# Flags and filenames
save_flag = True                                                # Save the agent checkpoint
load_flag = False                                               # Load the agent checkpoint 
train_flag = True                                               # Train the agent
save_sheet_flag = True                                          # Save the configuration to an Excel sheet

save_filename = timestamp + '.pth'                              # Filename for saving the agent checkpoint
load_filename = ''                                              # Filename for loading the agent checkpoint
file_name = f"{task_name}.xlsx"             # Filename for saving the Excel sheet

In [7]:
# %%
from RL4CRN.utils.visualizations import plot_truth_table

# Hyperparameters
max_added_reactions = 5                             # Maximum number of reactions
N_CPUs = os.cpu_count()                             # Number of CPUs          
N = 10*N_CPUs                                       # Number of samples (batch size)    
width = 1024                                        # Width of the neural networks  
depth = 5                                           # Depth of the neural networks 
deep_layer_size = 1024*10                           # Size of the deep layer encoding the CRNs
allow_input_influence = False                       # Allow input influence in the policy
learning_rate = 1e-4                                # Learning rate for the optimizer 
hall_of_fame_size = 30                              # Size of the hall of fame  
entropy_scheduler = {                               # Entropy scheduler parameters 
    'entropy_weight': 1e-3, 
    'topk_entropy_weight' : 1.0,
    'remainder_entropy_weight' : 1.0,
    'entropy_update_coefficient': 1, 
    'entropy_schedule': 1000, 
    'minimum_entropy_weight': 0.0
}
entropy_weights_per_head = {'structure': 2.0, 'continuous': 1.0, 'discrete': 0.0, 'input_influence': 0.0} 
structure_head_temperature = {"target_entropy_ratio_to_max": np.log(5)/np.log(M), "initial_temperature": 1.0, "rate": 0.0, "current_temperature": 1.0}
risk_scheduler = {                                  # Risk scheduler parameters
    'risk': 0.95, 
    'risk_update': 0.0, 
    'max_risk': 1.0, 
    'risk_schedule': 1000
}
epoch_num = 300                                     # Number of epochs for training
render_schedule = 10                                 # Render every # of epochs
render_mode = {                                     # Mode of the experiment
    'style': 'logger', 
    'task': 'transients', 
    'format': 'image',
    'topology': True,
    'bounds': [2.5]
}
# Ordering specific parameters
ordering_parameters = {
    'enforce_ordering': False,
    'constraint_weight' : float('inf')
}
# SIL settings
sil_settings = {
    'sil_loss_weight': 1.0,
    'sil_use_adaptive_baseline': False,
    'sil_baseline_annealing_rate': 0.95
}

render_n_best = 10                                                    # Number of best CRNs to plot responses for
render_disregard_percentage = 0.99                                    # Percentage of worst CRNs to disregard in the responses plotting

# Parameter distribution for the reactions added by the agent
continuous_distribution = {"type": 'lognormal_1D'}

# Time horizon for the simulation (we repeat this twice)
t_f = 100                                           # Final time for the simulation
N_t = 1000                                          # Number of time steps
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Construct the IOCRN inputs
# all combination of inputs between 0 and 1 
nums = [0., 1.]

# at t0, S=1 and while Mem and X can be in {0,1}
u_list_t0 = [np.array([u[0], 1, u[1]]) for u in product(*[nums for _ in range(2)])] # list of input combinations, each input is a numpy array of shape (p,)
# at t1, S=0 and while Mem and X can be in {0,1}
u_list_t1 = [np.array([u[0], 0, u[1]]) for u in product(*[nums for _ in range(2)])] # list of input combinations, each input is a numpy array of shape (p,)

# Construct the IOCRN initial conditions
ic = IC(names=species_labels, values=[[0.0 for _ in species_labels]])

# Construct the weights for the performance metric
# w = np.ones(N_t)
# w[(len(w)//5)*4:] = w[(len(w)//5)*4:]*2
# w[:(len(w)//5)] = w[:(len(w)//5)]*0.25
# w = w[np.newaxis, :]
# No transients
w = np.zeros((1, 2*N_t))
w[:, -1] = 1.0 * N_t

# 1. Zip inputs to create pairs: [[u_t0_c1, u_t1_c1], [u_t0_c2, u_t1_c2], ...]
# u_sequence_list = list(zip(u_list_t0, u_list_t1))
# do the prouct instead to have all combinations
u_sequence_list = []
for u0 in u_list_t0:
    for u1 in u_list_t1:
        u_sequence_list.append([u0, u1])

r_list = []
for u_seq in u_sequence_list:
    # when set=1, output should be X
    if u_seq[0][2] == 1: # identity
        r_list.append(np.array([u_seq[1][0]]))
    else: # NOT
        r_list.append(np.array([1 - u_seq[1][0]]))

print("Number of scenarios:", len(u_sequence_list))
print("Input sequences:", u_sequence_list)
print("Desired outputs:", r_list)

def compute_reward(state):
    x0_list = ic.get_ic(state)
    
    # 2. FIX: Create matching time horizons structure
    time_horizons_list = [time_horizon, time_horizon] 

    loss = dynamic_tracking_error_piecewise(
        state, 
        u_sequence_list,       
        x0_list, 
        time_horizons_list, 
        r_list,                
        w, 
        norm=1, 
        LARGE_NUMBER=1e4
    )
    return loss

Number of scenarios: 16
Input sequences: [[array([0., 1., 0.]), array([0., 0., 0.])], [array([0., 1., 0.]), array([0., 0., 1.])], [array([0., 1., 0.]), array([1., 0., 0.])], [array([0., 1., 0.]), array([1., 0., 1.])], [array([0., 1., 1.]), array([0., 0., 0.])], [array([0., 1., 1.]), array([0., 0., 1.])], [array([0., 1., 1.]), array([1., 0., 0.])], [array([0., 1., 1.]), array([1., 0., 1.])], [array([1., 1., 0.]), array([0., 0., 0.])], [array([1., 1., 0.]), array([0., 0., 1.])], [array([1., 1., 0.]), array([1., 0., 0.])], [array([1., 1., 0.]), array([1., 0., 1.])], [array([1., 1., 1.]), array([0., 0., 0.])], [array([1., 1., 1.]), array([0., 0., 1.])], [array([1., 1., 1.]), array([1., 0., 0.])], [array([1., 1., 1.]), array([1., 0., 1.])]]
Desired outputs: [array([1.]), array([1.]), array([0.]), array([0.]), array([0.]), array([0.]), array([1.]), array([1.]), array([1.]), array([1.]), array([0.]), array([0.]), array([0.]), array([0.]), array([1.]), array([1.])]


In [8]:
if save_sheet_flag:
    sheet_name = "Data"
    headers = [
        "Timestamp", "URL",
        "Epochs Completed", "Successful", "Saved", "Comments",
        "Learning Rate", "Epochs #",
        "(m, n, p, N)",
        "NN Depth", "NN Width", "Deep Layer Size", "CPUs #",
        "Entropy Scheduler",
        "Risk Scheduler",
        "Render Schedule", "HoF Size",
        "Simulation Time", "Time Steps #",
        "Initial Conditions #", "Input Scenarios#",
        "Continuous Distribution", "Entropy Weights per Head",
        "Structure Head Temperature",
        "Ordering Enforced",
        "SIL Settings"
    ]

    data_row = [
        timestamp, logger.url,
        None, None, None, None,
        learning_rate, epoch_num,
        str((max_added_reactions, len(species_labels), p, N)),
        depth, width, deep_layer_size, N_CPUs,
        str(entropy_scheduler),
        str(risk_scheduler),
        render_schedule, hall_of_fame_size,
        t_f, N_t, len(ic.values), len(u_list_t0),
        str(continuous_distribution), str(entropy_weights_per_head),
        str(structure_head_temperature),
        f"Yes: {ordering_parameters['constraint_weight']}" if ordering_parameters['enforce_ordering'] else "No",
        str(sil_settings)
    ]

    if os.path.exists(file_name):
        wb = load_workbook(file_name)
        if sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
        else:
            ws = wb.create_sheet(sheet_name)
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = sheet_name

    # Write headers if sheet is empty
    if ws.max_row == 1 and ws.max_column == 1 and ws.cell(row=1, column=1).value is None:
        for col, header in enumerate(headers, start=1):
            ws.cell(row=1, column=col, value=header)

    # Append experiment as next row
    next_row = ws.max_row + 1
    for col, value in enumerate(data_row, start=1):
        ws.cell(row=next_row, column=col, value=value)

    # Freeze header row and add filter
    ws.freeze_panes = "B1" 
    ws.auto_filter.ref = ws.dimensions

    # === Auto-fit column widths (except URL column) ===
    # URL column is column 2 (B), we leave its width unchanged.
    url_col_index = 2

    for col in range(1, ws.max_column + 1):
        if col == url_col_index:
            continue  # keep URL column width as-is

        max_length = 0
        for row in range(1, ws.max_row + 1):
            cell = ws.cell(row=row, column=col)
            value = cell.value
            if value is not None:
                # Convert to string to measure length
                length = len(str(value))
                if length > max_length:
                    max_length = length

        # Some padding so text isn't touching the cell border
        adjusted_width = max_length + 2 if max_length > 0 else 10
        col_letter = get_column_letter(col)
        ws.column_dimensions[col_letter].width = adjusted_width

    wb.save(file_name)
    print(f"New experiment data saved in row {next_row} of '{file_name}'.")

New experiment data saved in row 15 of 'MemoryCircuits_4s_5r_MAK_REINFORCE_SIL.xlsx'.


In [9]:
# Construct parallel environments
crn_0 = crn_template.clone()
mult_env = ParallelEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, N_CPUs=N_CPUs, logger=logger)
# mult_env = SerialEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, logger=logger)

In [10]:
# Construct the policy
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_head_attributes = {"hidden_size": width, "num_layers": depth}
rate_head_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_head_attributes = {"hidden_size": width, "num_layers": depth}
masks = {"continuous": library.get_parameter_mask(mode="continuous"), "discrete": library.get_parameter_mask(mode="discrete"), "logit": library.get_logit_mask()}
policy = AddReactionByOrderedIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, target_set_size=crn_template.num_reactions+max_added_reactions, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head,
                                    combinatorial_bias_enabled=ordering_parameters["enforce_ordering"], constraint_strength=ordering_parameters["constraint_weight"])

if ordering_parameters["enforce_ordering"]:
    policy = AddReactionByOrderedIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, target_set_size=crn_template.num_reactions+max_added_reactions, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head,
                                        combinatorial_bias_enabled=ordering_parameters["enforce_ordering"], constraint_strength=ordering_parameters["constraint_weight"])
else:
    policy = AddReactionByIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head)

# Construct the agent
agent = REINFORCEAgent(policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_scheduler=entropy_scheduler, risk_scheduler=risk_scheduler, sil_settings=sil_settings, device=device)
if load_flag:
    agent.policy.load_state_dict(torch.load(load_filename+'.pth', map_location=device))

In [11]:
# Construct the interfaces
observer = ExplicitObserver(reaction_library=library, allow_input_observation=allow_input_influence)
tensorizer = ExplicitTensorizer(device=device)
actuator = LibraryActuator(reaction_library=library)
stepper = IOCRNStepper()

In [12]:
# Training Loop   
if train_flag:
    agent.policy.train()
    for i in tqdm(range(epoch_num)):
        mult_env.reset()
        for j in range(max_added_reactions):
            observations = mult_env.observe(observer, tensorizer)
            actions, raw_actions = agent.act(observations, actuator)
            out = mult_env.step(actions, stepper, raw_actions=raw_actions)
        rewards = mult_env.get_reward(compute_reward)
        agent.update(rewards, step_iteration=i, hof=mult_env.hall_of_fame, observer=observer, tensorizer=tensorizer, stepper=stepper, use_sil=True, sil_weighting_scheme='uniform', sil_batch_size=None)
        if i % render_schedule == 0:
            mult_env.render(rewards, n_best=render_n_best, disregarded_percentage=render_disregard_percentage, mode=render_mode)

  0%|          | 0/300 [00:00<?, ?it/s]


[cvHandleFailure, Error: -15] At t = 37.9825855130734, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 3.54829480582165, unable to satisfy inequality constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvHandleFailure, Error: -15] At t = 14.0046458656579, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 50.5802792636952, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 14.0111568985477, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 44.5331072928561, unable to satisfy inequality constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Erro

  0%|          | 1/300 [00:47<3:57:35, 47.68s/it]


[cvHandleFailure, Error: -15] At t = 22.503040782777, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 10.6536198372035, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 20.3673153633478, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 15.2978941322169, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 19.8627023631685, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 14.4129773124533, unable to satisfy inequality constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvHandleFailure, Error: -15] At t = 4.14805670840089, unable to satisfy inequality constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy co

  1%|          | 2/300 [01:21<3:17:41, 39.80s/it]

...

 99%|█████████▉| 298/300 [3:21:13<01:19, 39.96s/it]


[cvHandleFailure, Error: -15] At t = 2.76786240082016, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 1.8287982356221, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 14.5520900243367, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 6.76722606376304, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 7.53976956767457, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 16.2404140201862, unable to satisfy inequality constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvHandleFailure, Error: -15] At t = 18.4817840174646, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 16.15277701260

100%|█████████▉| 299/300 [3:21:51<00:39, 39.55s/it]


[cvHandleFailure, Error: -15] At t = 1.26582826406565, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 1.99243471078259, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 10.179134092045, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 12.3478278270544, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 2.13461668718177, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 5.02795393516069, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 1.53481988642881, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 3.77421858133412, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 29.0341320867963, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 7.94824856168879, unable to satisfy inequality constraints.


[cvInitial

100%|██████████| 300/300 [3:22:28<00:00, 40.49s/it]


In [13]:
hall_of_fame_crns = [env.state for env in mult_env.hall_of_fame]
if save_flag:
    if not os.path.exists('models'):
        os.makedirs('models')
    if not os.path.exists('hof'):
        os.makedirs('hof')
    torch.save(agent.policy.state_dict(), 'models/' + save_filename)
    torch.save(hall_of_fame_crns, 'hof/hall_of_fame_' + save_filename)